# Description

In this notebook, we benchmark EQL with division algorithm on the Korns benchmarks.

In [2]:
from __future__ import annotations

import math
from collections import namedtuple
from dataclasses import dataclass
from typing import Optional, Any, Dict, List

import numpy as np
import sympy as sp
import tensorflow as tf

from config.korns_config import BENCH, FEATURE_NAMES, EQLDIV
from src.korns_core import RunConfig, load_korns_hdf5, run_benchmark, SRFitResult

import src.EQLdiv.EQL_Layer_tf as eql
from src.EQLdiv.data_utils import get_penalty_data
from src.EQLdiv.evaluation import (
    calculate_complexity,
    symbolic_matmul_and_bias,
    symbolic_eql_layer,
    get_symbol_list,
    proper_simplify,
)
from src.EQLdiv.utils import get_div_thresh_fn, step_to_epochs


# -------------------------
# Division detection + GT lookup (no korns_core changes)
# -------------------------
def expr_has_division(expr: sp.Expr) -> bool:
    e = sp.sympify(expr)
    try:
        num, den = sp.fraction(sp.together(e))
        return den != 1
    except Exception:
        # conservative fallback: look for negative powers
        for p in e.atoms(sp.Pow):
            exp = p.exp
            if exp.is_Number:
                try:
                    if float(exp) < 0.0:
                        return True
                except Exception:
                    pass
        return False


# -------------------------
# Threshold penalty reduction (robust when collection is empty)
# -------------------------
def _sum_threshold_penalties() -> tf.Tensor:
    penalties = tf.compat.v1.get_collection("Threshold_penalties")
    if not penalties:
        return tf.constant(0.0, dtype=tf.float32)
    return tf.add_n([tf.reduce_sum(p) for p in penalties])


# -------------------------
# Symbolic utilities (no LaTeX, no file writes)
# -------------------------
def prune_small_coeff_add_terms(expr: sp.Expr, eps: float) -> sp.Expr:
    terms = sp.Add.make_args(expr)
    if len(terms) == 1:
        return expr

    kept = []
    for term in terms:
        c, _ = term.as_coeff_Mul()
        if c.is_number:
            try:
                if abs(float(c)) < eps:
                    continue
            except Exception:
                pass
        kept.append(term)

    if not kept:
        return sp.Integer(0)

    return sp.Add(*kept, evaluate=False)


def build_symbolic_outputs(
    kernels: List[np.ndarray],
    biases: List[np.ndarray],
    fns_list,
    *,
    round_decimals: int,
    simplify: bool,
) -> List[sp.Expr]:
    in_nodes = get_symbol_list(kernels[0].shape[0])
    res = in_nodes
    for kernel, bias, fns in zip(kernels, biases, fns_list):
        res = symbolic_matmul_and_bias(res, kernel, bias)
        res = symbolic_eql_layer(res, fns)

    out = []
    for e in res:
        e2 = e
        if round_decimals is not None:
            e2 = sp.N(e2, round_decimals)
        if simplify:
            try:
                e2 = proper_simplify(e2)
            except Exception:
                pass
        out.append(e2)
    return out


# -------------------------
# EQL-div model (self-contained, no file-based metadata)
# -------------------------
class Model(object):
    def __init__(
        self,
        *,
        mode: str,
        metadata: Dict[str, Any],
        layer_width: int,
        num_h_layers: int,
        reg_sched: tuple[float, float],
        output_bound: Optional[float],
        weight_init_param: float,
        epoch_factor: int,
        batch_size: int,
        test_div_threshold: float,
        reg_scale: float,
        l0_threshold: float,
        train_val_split: float,
        network_init_seed: Optional[int] = None,
        layer_ops: Optional[List[str]] = None,
        out_op: str = "reg_div",
    ):
        self.metadata = metadata
        self.train_data_size = int(train_val_split * metadata["train_val_examples"])
        self.width = int(layer_width)
        self.num_h_layers = int(num_h_layers)
        self.weight_init_scale = float(weight_init_param) / math.sqrt(metadata["num_inputs"] + num_h_layers)
        self.seed = network_init_seed

        self.reg_start = math.floor(num_h_layers * epoch_factor * reg_sched[0])
        self.reg_end = math.floor(num_h_layers * epoch_factor * reg_sched[1])

        self.output_bound = float(output_bound if output_bound is not None else metadata["extracted_output_bound"])
        self.reg_scale = float(reg_scale)
        self.batch_size = int(batch_size)
        self.l0_threshold = float(l0_threshold)
        self.is_training = (mode == "train")

        allowed_ops = {"multiply", "sin", "cos", "id", "sub", "log", "exp", "reg_div"}

        if layer_ops is None:
            layer_ops = ["sin", "cos", "multiply", "id"]
        layer_ops = list(layer_ops)

        bad = set(layer_ops) - allowed_ops
        if bad:
            raise ValueError(f"Unknown ops in layer_ops: {sorted(bad)}")
        if "reg_div" in layer_ops:
            raise ValueError("reg_div is only supported as out_op, not in layer_ops")

        hidden_kwargs = {op: self.width for op in layer_ops}

        div_thresh_fn = get_div_thresh_fn(
            self.is_training,
            self.batch_size,
            test_div_threshold,
            train_examples=self.train_data_size,
        )
        reg_div = namedtuple("reg_div", ["repeats", "div_thresh_fn"])

        self.eql_layers = [
            eql.EQL_Layer(**hidden_kwargs, weight_init_scale=self.weight_init_scale, seed=self.seed)
            for _ in range(self.num_h_layers)
        ]

        if out_op not in allowed_ops:
            raise ValueError(f"Unknown out_op: {out_op}")

        if out_op == "reg_div":
            self.eql_layers.append(
                eql.EQL_Layer(
                    reg_div=reg_div(repeats=metadata["num_outputs"], div_thresh_fn=div_thresh_fn),
                    weight_init_scale=self.weight_init_scale,
                    seed=self.seed,
                )
            )
        else:
            self.eql_layers.append(
                eql.EQL_Layer(
                    **{out_op: metadata["num_outputs"]},
                    weight_init_scale=self.weight_init_scale,
                    seed=self.seed,
                )
            )

    def __call__(self, inputs, global_step):
        num_epochs = step_to_epochs(global_step, self.batch_size, self.train_data_size)

        l1_mask = tf.cast(tf.less(num_epochs, self.reg_end), tf.float32) * tf.cast(
            tf.greater(num_epochs, self.reg_start), tf.float32
        )
        l1_reg_sched = l1_mask * tf.constant(self.reg_scale, dtype=tf.float32)

        l0_threshold = tf.cond(
            tf.less(num_epochs, self.reg_end),
            lambda: tf.constant(0.0, dtype=tf.float32),
            lambda: tf.constant(self.l0_threshold, dtype=tf.float32),
        )

        output = inputs
        for layer in self.eql_layers:
            output = layer(output, l1_reg_sched=l1_reg_sched, l0_threshold=l0_threshold)

        P_bound = (tf.abs(output) - self.output_bound) * tf.cast(
            (tf.abs(output) > self.output_bound), dtype=tf.float32
        )
        return output, P_bound, l1_reg_sched


def _make_optimizer(lr: float, beta1: float):
    if hasattr(tf.keras.optimizers, "legacy"):
        return tf.keras.optimizers.legacy.Adam(learning_rate=lr, beta_1=beta1)
    return tf.keras.optimizers.Adam(learning_rate=lr, beta_1=beta1)


def _mse(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))


def _l1_from_model(model: Model):
    terms = []
    for layer in model.eql_layers:
        if hasattr(layer, "_dense") and layer._dense is not None:
            terms.append(tf.reduce_sum(tf.abs(layer._dense.kernel)))
            terms.append(tf.reduce_sum(tf.abs(layer._dense.bias)))
    if not terms:
        return tf.constant(0.0, dtype=tf.float32)
    return tf.add_n(terms)


def _extract_kernels_biases_in_order(model: Model):
    kernels, biases = [], []
    for layer in model.eql_layers:
        if not hasattr(layer, "_dense") or layer._dense is None:
            raise ValueError("EQL_Layer has no built Dense yet (layer._dense is None).")
        kernels.append(layer._dense.kernel.numpy())
        biases.append(layer._dense.bias.numpy())
    return kernels, biases


def _build_dataset_from_numpy(X, y, *, batch_size: int, repeats: int, shuffle: bool):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10_000, int(X.shape[0])))
    ds = ds.repeat(repeats).batch(batch_size, drop_remainder=False)
    return ds


# -------------------------
# Benchmark wrapper (params come from EQLDIV)
# -------------------------
@dataclass
class EQLDivKornsRegressor:
    name: str = EQLDIV.name

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        expr_gt = EXPR_BY_X_ID.get(id(X_train), None)
        out_op = "reg_div" if (expr_gt is not None and expr_has_division(expr_gt)) else "id"
        print("output op", out_op)

        X_train = np.asarray(X_train, dtype=np.float32)
        X_test = np.asarray(X_test, dtype=np.float32)
        y_train = np.asarray(y_train, dtype=np.float32).reshape(-1, 1)

        n_train = int(X_train.shape[0])
        num_inputs = int(X_train.shape[1])
        num_outputs = int(y_train.shape[1])

        # penalty bounds from training box
        xmins = X_train.min(axis=0)
        xmaxs = X_train.max(axis=0)
        penalty_bounds = [(float(lo), float(hi)) for lo, hi in zip(xmins, xmaxs)]

        y_abs = np.abs(y_train.reshape(-1))
        extracted_output_bound = float(max(1.0, np.quantile(y_abs, 0.99) * 2.0))

        metadata = {
            "train_val_examples": n_train,
            "num_inputs": num_inputs,
            "num_outputs": num_outputs,
            "extracted_output_bound": extracted_output_bound,
            "extracted_penalty_bounds": penalty_bounds,
        }

        # penalty data (capped for speed)
        n_pen = int(min(EQLDIV.penalty_examples_cap, n_train))
        X_pen, y_pen = get_penalty_data(
            num_examples=n_pen,
            penalty_bounds=penalty_bounds,
            num_inputs=num_inputs,
            num_outputs=num_outputs,
        )
        X_pen = np.asarray(X_pen, dtype=np.float32)
        y_pen = np.asarray(y_pen, dtype=np.float32).reshape(-1, 1)

        model = Model(
            mode="train",
            metadata=metadata,
            layer_width=EQLDIV.layer_width,
            num_h_layers=EQLDIV.num_h_layers,
            reg_sched=EQLDIV.reg_sched,
            output_bound=EQLDIV.output_bound,
            weight_init_param=EQLDIV.weight_init_param,
            epoch_factor=EQLDIV.epoch_factor,
            batch_size=EQLDIV.batch_size,
            test_div_threshold=EQLDIV.test_div_threshold,
            reg_scale=EQLDIV.reg_scale,
            l0_threshold=EQLDIV.l0_threshold,
            train_val_split=1.0,
            network_init_seed=None,
            layer_ops=list(EQLDIV.layer_ops),
            out_op=out_op,
        )

        optimizer = _make_optimizer(EQLDIV.learning_rate, EQLDIV.beta1)
        global_step = tf.Variable(0, dtype=tf.int64, trainable=False)

        def train_step(xb, yb, use_penalty: bool):
            tf.compat.v1.get_collection_ref("Threshold_penalties").clear()
            with tf.GradientTape() as tape:
                preds, P_bound, l1_reg_sched = model(xb, global_step)

                bound_penalty = tf.reduce_sum(P_bound)
                P_theta = _sum_threshold_penalties()

                mse_loss = _mse(yb, preds)
                l1_loss = tf.cast(l1_reg_sched, tf.float32) * _l1_from_model(model)

                penalty_loss = P_theta + bound_penalty + l1_loss
                normal_loss = mse_loss + P_theta + l1_loss
                loss = penalty_loss if use_penalty else normal_loss

            vars_ = tape.watched_variables()
            grads = tape.gradient(loss, vars_)
            optimizer.apply_gradients(zip(grads, vars_))
            global_step.assign_add(1)

        # training: episodes = epoch_factor (mirrors your current behavior)
        for _episode in range(EQLDIV.epoch_factor):
            ds_pen = _build_dataset_from_numpy(
                X_pen, y_pen, batch_size=EQLDIV.batch_size, repeats=1, shuffle=True
            )
            for xb, yb in ds_pen:
                train_step(xb, yb, True)

            ds_train = _build_dataset_from_numpy(
                X_train, y_train, batch_size=EQLDIV.batch_size, repeats=EQLDIV.penalty_every, shuffle=True
            )
            for xb, yb in ds_train:
                train_step(xb, yb, False)

        # predictions
        y_pred_train = model(tf.convert_to_tensor(X_train), global_step)[0].numpy().reshape(-1)
        y_pred_test = model(tf.convert_to_tensor(X_test), global_step)[0].numpy().reshape(-1)

        # symbolic + complexity
        kernels, biases = _extract_kernels_biases_in_order(model)
        fns_list = [layer.get_fns() for layer in model.eql_layers]

        complexity = calculate_complexity(
            kernels, biases, fns_list, thresh=float(EQLDIV.complexity_threshold)
        )

        exprs = build_symbolic_outputs(
            kernels=kernels,
            biases=biases,
            fns_list=fns_list,
            round_decimals=int(EQLDIV.round_decimals),
            simplify=bool(EQLDIV.simplify),
        )
        expr0 = exprs[0]
        if float(EQLDIV.symbolic_prune_threshold) > 0:
            expr0 = prune_small_coeff_add_terms(expr0, float(EQLDIV.symbolic_prune_threshold))

        return SRFitResult(
            expr=expr0,
            y_pred_train=np.asarray(y_pred_train, dtype=np.float64),
            y_pred_test=np.asarray(y_pred_test, dtype=np.float64),
            metadata={"complexity": float(complexity), "out_op": out_op},
        )


# -------------------------
# Run benchmark (experiment params from BENCH only)
# -------------------------
cfg = RunConfig(
    hdf5_path=BENCH.hdf5_path,
    test_size=BENCH.test_size,
    split_seed=BENCH.split_seed,
    per_problem_seed_offset=BENCH.per_problem_seed_offset,
    algo_seed_offset=BENCH.algo_seed_offset,
    run_seed_offset=BENCH.run_seed_offset,
)

datasets = load_korns_hdf5(cfg.hdf5_path)

EXPR_BY_X_ID = {id(rec.X_train): rec.expr_gt for rec in datasets.values()}

rows = run_benchmark(
    datasets=datasets,
    algorithms=[EQLDivKornsRegressor()],
    config=cfg,
    n_runs=BENCH.n_runs,
    feature_names=FEATURE_NAMES,
    results_csv_path=EQLDIV.results_csv_path,
)


[PROBLEM] P1
[GT] 24.3*x3 + 1.57
[ALGO] eql_div
[RUN START] run_id=0 seed=1630210511000
output op id
[PRED] 0.151692*x_1 + 0.1543*x_2 - 0.354307*x_3 + 1.63752*x_4 + 0.109009*x_5 - 0.364785*(Abs(0.204119548201561*x_1 - 0.151973351836205*x_3 + 0.298191428184509*x_4 - 0.0239162519574165*x_5) + 1.0e-12)**0.142755*exp(-0.0297758450916508*x_1 - 0.0302878049730615*x_2 + 0.0695475909357191*x_3 - 0.321431195791444*x_4 - 0.0213975195940486*x_5 + 0.126831978559494*(0.708506286144257*x_4 - 0.374737203121185)*(0.566810131072998*x_4 + 0.0243563745170832*x_5 - 0.361718744039536) - 0.0702024772763252*sin(0.106904163956642*x_1 + 0.117630712687969*x_2 - 0.0662478804588318*x_3 - 0.269357740879059*x_4 + 0.0656358972191811*x_5) + 0.241301253437996*cos(-0.0534081496298313*x_1 + 0.0300821661949158*x_2 - 0.0537694096565247*x_3 + 0.0950709581375122*x_4 + 0.134231880307198*x_5)) - 1.32018*(-0.10321*x_1 - 0.104985*x_2 + 0.241069*x_3 - 1.11416*x_4 - 0.0741689*x_5 - 0.838408*log(Abs(0.204119548201561*x_1 - 0.15197

KeyboardInterrupt: 